# DataFrame Mental Model

In [1]:
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

spark = SparkSession.builder.appName("module-03-dataframes").master("local[*]").getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "4")
base_02 = "../../datasets/module_02"
base_03 = "../../datasets/module_03"

In [2]:
municipalities_schema = StructType([
    StructField("municipality_id", IntegerType(), True),
    StructField("municipality_name", StringType(), True),
    StructField("canton", StringType(), True),
    StructField("population", IntegerType(), True),
])
accessibility_schema = StructType([
    StructField("municipality_id", IntegerType(), True),
    StructField("accessibility_score", DoubleType(), True),
])
poi_schema = StructType([
    StructField("municipality_id", IntegerType(), True),
    StructField("poi_count", IntegerType(), True),
])
property_schema = StructType([
    StructField("municipality_id", IntegerType(), True),
    StructField("property_value_index", DoubleType(), True),
])
transactions_schema = StructType([
    StructField("transaction_id", IntegerType(), True),
    StructField("municipality_id", IntegerType(), True),
    StructField("property_id", IntegerType(), True),
    StructField("sale_price", IntegerType(), True),
    StructField("sale_date", StringType(), True),
    StructField("property_type", StringType(), True),
])
pop_history_schema = StructType([
    StructField("municipality_id", IntegerType(), True),
    StructField("year", IntegerType(), True),
    StructField("population", IntegerType(), True),
])

municipalities = spark.read.option("header", True).schema(municipalities_schema).csv(f"{base_02}/municipalities.csv")
accessibility_scores = spark.read.option("header", True).schema(accessibility_schema).csv(f"{base_02}/accessibility_scores.csv")
poi_counts = spark.read.option("header", True).schema(poi_schema).csv(f"{base_02}/poi_counts.csv")
property_values = spark.read.option("header", True).schema(property_schema).csv(f"{base_02}/property_values.csv")
transactions = spark.read.option("header", True).schema(transactions_schema).csv(f"{base_03}/transactions.csv").withColumn("sale_date", F.to_date("sale_date"))
population_history = spark.read.option("header", True).schema(pop_history_schema).csv(f"{base_03}/population_history.csv")

for name, df in {
    "municipalities": municipalities,
    "accessibility_scores": accessibility_scores,
    "poi_counts": poi_counts,
    "property_values": property_values,
    "transactions": transactions,
    "population_history": population_history,
}.items():
    df.createOrReplaceTempView(name)

In [3]:
# SQL and DataFrame versions of the same projection
spark.sql("SELECT municipality_id, municipality_name, canton FROM municipalities").show(5)
municipalities.select("municipality_id", "municipality_name", "canton").show(5)
municipalities.select("municipality_id", "municipality_name", "canton").explain("formatted")

+---------------+-----------------+------+
|municipality_id|municipality_name|canton|
+---------------+-----------------+------+
|              1|           Zurich|    ZH|
|              2|       Winterthur|    ZH|
|              3|            Uster|    ZH|
|              4|           Meilen|    ZH|
|              5|             Bern|    BE|
+---------------+-----------------+------+
only showing top 5 rows

+---------------+-----------------+------+
|municipality_id|municipality_name|canton|
+---------------+-----------------+------+
|              1|           Zurich|    ZH|
|              2|       Winterthur|    ZH|
|              3|            Uster|    ZH|
|              4|           Meilen|    ZH|
|              5|             Bern|    BE|
+---------------+-----------------+------+
only showing top 5 rows

== Physical Plan ==
Scan csv  (1)


(1) Scan csv 
Output [3]: [municipality_id#0, municipality_name#1, canton#2]
Batched: false
Location: InMemoryFileIndex [file:/home/jovyan/w

In [4]:
municipalities.select(
    "municipality_id",
    "municipality_name",
    "canton"
).explain("formatted")

== Physical Plan ==
Scan csv  (1)


(1) Scan csv 
Output [3]: [municipality_id#0, municipality_name#1, canton#2]
Batched: false
Location: InMemoryFileIndex [file:/home/jovyan/work/datasets/module_02/municipalities.csv]
ReadSchema: struct<municipality_id:int,municipality_name:string,canton:string>




In [5]:
residential = transactions.select(
    "transaction_id",
    "municipality_id",
    "sale_price",
    "sale_date",
    "property_type",
).filter(
    (F.col("property_type").isin("apartment", "detached_house", "row_house"))
    & (F.col("sale_price") >= 500000)
)


In [6]:
residential.show()

+--------------+---------------+----------+----------+--------------+
|transaction_id|municipality_id|sale_price| sale_date| property_type|
+--------------+---------------+----------+----------+--------------+
|            20|             14|    566000|2024-08-12|detached_house|
|            30|              1|    537000|2024-11-10|detached_house|
|            51|             14|    612000|2025-05-13|detached_house|
|            77|              1|    799000|2025-02-25|detached_house|
|           120|              1|    854000|2024-12-24|detached_house|
+--------------+---------------+----------+----------+--------------+



In [7]:
residential.explain("formatted")

== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- Scan csv  (1)


(1) Scan csv 
Output [5]: [transaction_id#20, municipality_id#21, sale_price#23, sale_date#24, property_type#25]
Batched: false
Location: InMemoryFileIndex [file:/home/jovyan/work/datasets/module_03/transactions.csv]
PushedFilters: [IsNotNull(sale_price), In(property_type, [apartment,detached_house,row_house]), GreaterThanOrEqual(sale_price,500000)]
ReadSchema: struct<transaction_id:int,municipality_id:int,sale_price:int,sale_date:string,property_type:string>

(2) Filter [codegen id : 1]
Input [5]: [transaction_id#20, municipality_id#21, sale_price#23, sale_date#24, property_type#25]
Condition : ((isnotnull(sale_price#23) AND property_type#25 IN (apartment,detached_house,row_house)) AND (sale_price#23 >= 500000))

(3) Project [codegen id : 1]
Output [5]: [transaction_id#20, municipality_id#21, sale_price#23, cast(sale_date#24 as date) AS sale_date#32, property_type#25]
Input [5]: [transaction_id#20, municipality_id

In [8]:
transactions_enriched = (
    transactions
    .withColumn("sale_year", F.year("sale_date"))
    .withColumn("sale_price_millions", F.round(F.col("sale_price") / F.lit(1000000.0), 3))
    .withColumn(
        "price_band",
        F.when(F.col("sale_price") >= 1000000, F.lit("premium"))
         .when(F.col("sale_price") >= 500000, F.lit("standard"))
         .otherwise(F.lit("entry"))
    )
)

In [9]:
transactions_enriched.show()

+--------------+---------------+-----------+----------+----------+--------------+---------+-------------------+----------+
|transaction_id|municipality_id|property_id|sale_price| sale_date| property_type|sale_year|sale_price_millions|price_band|
+--------------+---------------+-----------+----------+----------+--------------+---------+-------------------+----------+
|             1|             21|      10001|    336000|2024-09-07|detached_house|     2024|              0.336|     entry|
|             2|              5|      10002|    596000|2024-03-30|     mixed_use|     2024|              0.596|  standard|
|             3|             14|      10003|    265000|2024-08-26|     mixed_use|     2024|              0.265|     entry|
|             4|             20|      10004|    247000|2025-03-05|detached_house|     2025|              0.247|     entry|
|             5|             15|      10005|    371000|2024-01-07|detached_house|     2024|              0.371|     entry|
|             6|

In [10]:
transactions_enriched.explain("formatted")

== Physical Plan ==
* Project (3)
+- * Project (2)
   +- Scan csv  (1)


(1) Scan csv 
Output [6]: [transaction_id#20, municipality_id#21, property_id#22, sale_price#23, sale_date#24, property_type#25]
Batched: false
Location: InMemoryFileIndex [file:/home/jovyan/work/datasets/module_03/transactions.csv]
ReadSchema: struct<transaction_id:int,municipality_id:int,property_id:int,sale_price:int,sale_date:string,property_type:string>

(2) Project [codegen id : 1]
Output [6]: [transaction_id#20, municipality_id#21, property_id#22, sale_price#23, cast(sale_date#24 as date) AS sale_date#32, property_type#25]
Input [6]: [transaction_id#20, municipality_id#21, property_id#22, sale_price#23, sale_date#24, property_type#25]

(3) Project [codegen id : 1]
Output [9]: [transaction_id#20, municipality_id#21, property_id#22, sale_price#23, sale_date#32, property_type#25, year(sale_date#32) AS sale_year#122, round((cast(sale_price#23 as double) / 1000000.0), 3) AS sale_price_millions#130, CASE WHEN (sa

In [11]:
summary = transactions_enriched.groupBy(
    "municipality_id",
    "property_type"
).agg(
    F.count("*").alias("transaction_count"),
    F.avg("sale_price").alias("avg_sale_price"),
    F.max("sale_price").alias("max_sale_price"),
    F.sum("sale_price").alias("total_sale_value"),
)

In [12]:
summary.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- HashAggregate (2)
         +- Scan csv  (1)


(1) Scan csv 
Output [3]: [municipality_id#21, sale_price#23, property_type#25]
Batched: false
Location: InMemoryFileIndex [file:/home/jovyan/work/datasets/module_03/transactions.csv]
ReadSchema: struct<municipality_id:int,sale_price:int,property_type:string>

(2) HashAggregate
Input [3]: [municipality_id#21, sale_price#23, property_type#25]
Keys [2]: [municipality_id#21, property_type#25]
Functions [4]: [partial_count(1), partial_avg(sale_price#23), partial_max(sale_price#23), partial_sum(sale_price#23)]
Aggregate Attributes [5]: [count#215L, sum#216, count#217L, max#218, sum#219L]
Results [7]: [municipality_id#21, property_type#25, count#220L, sum#221, count#222L, max#223, sum#224L]

(3) Exchange
Input [7]: [municipality_id#21, property_type#25, count#220L, sum#221, count#222L, max#223, sum#224L]
Arguments: hashpartitioning(municipality_id#21, prope

In [13]:
summary.show()

+---------------+--------------+-----------------+-----------------+--------------+----------------+
|municipality_id| property_type|transaction_count|   avg_sale_price|max_sale_price|total_sale_value|
+---------------+--------------+-----------------+-----------------+--------------+----------------+
|             21|detached_house|                3|357666.6666666667|        404000|         1073000|
|              3|     row_house|                1|         419000.0|        419000|          419000|
|             15|     row_house|                1|         294000.0|        294000|          294000|
|             13|     row_house|                3|312333.3333333333|        372000|          937000|
|             25|     mixed_use|                1|         351000.0|        351000|          351000|
|             20|     mixed_use|                4|         411000.0|        547000|         1644000|
|             25|    commercial|                3|355666.6666666667|        399000|        

In [14]:
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+

